# Diagnose an empty IPTW run

Companion to [05b_run_biomarker_pipeline.ipynb](05b_run_biomarker_pipeline.ipynb), for when stage 5
finishes without error but the CSVs in `biomarker_analysis/IPTW_runs_*/` have no rows.

### How a run finishes clean and still produces nothing

Every Track 1/2 fit starts with `filter_finite_rows(df.select(cols), cols)`, where `cols` is the
outcome columns plus `base_vars` plus the marker. `filter_finite_rows` casts each column to `Float64`
with `strict=False`, so a single **non-numeric** covariate becomes all-null, fails `is_finite()`, and
drops every row of the model frame. `CoxPHFitter` then raises for every marker in turn, `_safe_fit`
catches it, `results` comes back empty, and the screen writes a zero-row parquet. `compile_IPTW_results`
reads that without complaint and reports "0 significant hits" — so a broken run and a genuine null
result look identical on disk.

That is what the raw `CANCER_TYPE` label did: `build_cancer_type_df` deliberately keeps the string
column beside its dummies, and the pan-cancer branch selected covariates with a loose
`'CANCER_TYPE' in c` substring test, which swept it into `base_vars`. The covariate filters are now
anchored on the dummy prefixes (`CANCER_TYPE_`, `PANEL_VERSION_`), a numeric guard runs before each
screen, and a screen where every fit fails raises instead of writing an empty file.

### What this notebook does

1. **Guard tests** — the invariants that keep a silent empty run from recurring.
2. **Result inventory** — row counts, not byte sizes, for everything the last run wrote. A gzipped
   zero-row parquet still carries a schema and a footer, so `ls -l` looks perfectly plausible.
3. **Input diagnosis** — rebuilds `base_vars` from the saved `IPTW_df_*.parquet` exactly as
   `run_IPTW_analysis.main()` does, names any non-numeric covariate, attributes row loss per column,
   counts markers with support, and runs one trial fit with the traceback `_safe_fit` would have hidden.

Read-only throughout — nothing here writes to `DATA_PATH`.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find v2 root from {start}")


V2_ROOT = find_v2_root()
REPO_ROOT = V2_ROOT.parent
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

import config
import polars as pl

BIOMARKER_PATH = config.BIOMARKER_PATH

# Mirrors the grids in run_IPTW_analysis; change them in the script, not here.
COHORTS = ["cohort1", "cohort2"]
PS_MODELS = ["covariates_only", "covariates_plus_embeddings"]
SPECS = [f"{c}_{p}" for c in COHORTS for p in PS_MODELS]

print(f"v2 root:   {V2_ROOT}")
print(f"repo root: {REPO_ROOT}")
print(f"Python:    {sys.executable}")
print(f"Biomarker: {BIOMARKER_PATH}")

## 1. Guard tests

`tests/test_iptw_guards.py` pins the three invariants that make a silent empty run impossible:
the raw `CANCER_TYPE`/`PANEL_VERSION` string labels are rejected as model covariates, the prefix
filters that build `base_vars` exclude them, and `_run_marker_screen` raises when every fit fails
(while still tolerating partial failure and an empty marker list).

These `importorskip` on `statsmodels` and `zstandard`, so they **skip** in environments without
them — a "skipped" line here means the test never ran, not that it passed.

In [ ]:
proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "tests/test_iptw_guards.py"],
    cwd=REPO_ROOT,
)
if proc.returncode == 0:
    print("\nGuards hold.")
elif proc.returncode == 5:
    # pytest's NO_TESTS_COLLECTED: every test skipped, so nothing actually ran.
    print("\n[skipped] No tests collected — statsmodels/zstandard are missing from this "
          "environment, so the guards were never exercised. Run this on the cluster kernel.")
else:
    print(f"\n[FAIL] pytest exited {proc.returncode} — the guards are not in place; "
          "fix that before trusting anything below.")

## 2. What the last run wrote

Row counts for every result parquet already on disk. `EMPTY (no rows)` is the signature of a screen
whose fits all failed — the file exists, is valid gzip, has the right columns, and carries no data.

The per-cancer-type screens build `base_vars` without the cancer-type dummies, so if only the
`pan_cancer_*` rows are empty, the raw-label bug explains it completely. Empty per-type files mean
something else as well, and section 3 will say what.

In [ ]:
def result_rows(path: str) -> int | None:
    """Rows in a result parquet, from its metadata, or None if it cannot be read."""
    try:
        return pl.scan_parquet(path).select(pl.len()).collect().item()
    except (OSError, pl.exceptions.PolarsError):
        return None


total_files = 0
empty_files = 0

for spec in SPECS:
    run_path = os.path.join(BIOMARKER_PATH, f"IPTW_runs_{spec}/")
    print(f"\n{'=' * 78}\n{spec}\n{'=' * 78}")
    if not os.path.isdir(run_path):
        print("  no directory — stage 5 has not run for this spec")
        continue

    found = False
    for name in sorted(f for f in os.listdir(run_path) if f.endswith(".parquet")):
        found = True
        total_files += 1
        n_rows = result_rows(os.path.join(run_path, name))
        if n_rows is None:
            print(f"  {name:<58} UNREADABLE")
        elif n_rows == 0:
            empty_files += 1
            print(f"  {name:<58} {0:>6} rows   <- EMPTY (no rows)")
        else:
            print(f"  {name:<58} {n_rows:>6} rows")
    if not found:
        print("  directory exists but holds no .parquet — stage 5 stopped before writing")

print(f"\n{'=' * 78}")
if total_files == 0:
    print("No result files anywhere — stage 5 has not produced output yet. "
          "Section 3 still checks whether its inputs are sound.")
else:
    print(f"{empty_files}/{total_files} result files have no rows.")
if total_files and empty_files == total_files:
    print("Every screen failed. Section 3 names the covariate responsible.")
elif empty_files:
    print("Some screens produced results. Compare which cancer types are empty against "
          "the base_vars each branch builds.")

## 3. Input diagnosis

`pipelines.biomarkers.diagnose_iptw_inputs` reads the saved `IPTW_df_*.parquet` — no refit, no rerun,
seconds per spec — and for each specification reports:

- patient count, ICI/control split, and death count
- every non-numeric column in the frame
- the `pan_cancer` `base_vars` it would assemble, flagging any non-numeric member
- rows surviving `filter_finite_rows` on `base_vars` alone, **before** a marker is added — `0/N` here
  is the bug, stated outright
- rows lost per covariate in isolation, so the culprit is named even when several columns are
  numeric-but-null
- markers passing Track 1 / Track 2 support
- one real fit per track with `_safe_fit` removed, so the traceback is visible

Run it as a subprocess, matching 05b, so nothing leaks into this kernel.

Set `MAX_MARKERS` below for a fast pass: it caps the per-marker support scans, which are the
slow part (one numpy pass per marker, ~1500 markers x 2 tracks x 4 specs). Everything
structural — schema, rank, empty model frame, the trial fits — is marker-independent and still
runs in full. It is the same `IPTW_MAX_MARKERS` the analysis script uses, seeded identically,
so a capped diagnosis inspects exactly the markers a capped run screened.

In [ ]:
# Cap the per-marker support scans — the slow part of this script (a numpy pass
# per marker, ~1500 markers x 2 tracks x 4 specs). The structural checks (schema,
# rank, empty model frame) and the trial fits do not depend on the marker list,
# so a cap gives the same diagnosis in a fraction of the time.
#
# This is the same env var run_IPTW_analysis uses, and both scripts seed the RNG
# identically, so a cap here inspects exactly the markers a smoke run screened.
# None => scan every marker.
MAX_MARKERS: int | None = None   # e.g. 50 for a fast pass

DIAG_ENV = os.environ.copy()
if MAX_MARKERS is not None:
    DIAG_ENV["IPTW_MAX_MARKERS"] = str(MAX_MARKERS)
    print(f"Capping support scans at {MAX_MARKERS} markers per spec.\n")
else:
    # A stale value in the kernel's environment would silently cap this run and
    # make the support counts look wrong for reasons invisible in the notebook.
    DIAG_ENV.pop("IPTW_MAX_MARKERS", None)
    DIAG_ENV.pop("IPTW_MARKER_FRACTION", None)

proc = subprocess.run(
    [sys.executable, "-m", "pipelines.biomarkers.diagnose_iptw_inputs"],
    cwd=V2_ROOT, env=DIAG_ENV,
)
print(f"\n[exit {proc.returncode}]")

## 4. Track 1 null coefficients

For the case where Track 2 writes real hits but **Track 1 rows carry null/NaN `beta_marker`** —
a different failure from an empty screen. The fits did not raise (`_run_marker_screen` would have
stopped the run), so `_safe_fit` recorded each as a success carrying a non-finite coefficient.

Track 1 fits the **ICI arm alone**, so it has degeneracies Track 2 cannot have:

- a covariate that varies across the cohort but is **constant within the ICI arm** (a cancer type
  or line no ICI patient has) — the full-cohort constant-column drop in `main()` cannot see it
- `IPTW_GEN = 1 / ICI_prediction`, which inverts propensity scores **recalibrated within the
  subset**; near-zero scores blow the weights up before truncation, and `robust=True` then builds
  a sandwich estimator from them

This section answers three questions in order, each eliminating a different cause: are the nulls
on disk or created downstream, which covariates are degenerate on the ICI frame, and what does a
single **uncaught** fit actually return. Set `SPEC`/`CANCER` below to target one screen.

In [ ]:
# Track 1 null-coefficient diagnostic. Read-only: no refit of the screen, no writes.
import traceback

import numpy as np

from pipelines.biomarkers.run_IPTW_analysis import (
    IPTW_TRUNC_PCT,
    MIN_EVENTS_PER_MARKER_GROUP,
    MIN_MARKER_POS_ICI_ONLY,
    _fit_track1_marker,
    marker_has_ici_only_support,
)

SPEC = SPECS[0]          # e.g. "cohort1_covariates_only"
CANCER = "pan_cancer"    # any {cancer_type}_results.parquet in that run dir

# Reuses section 3's cap if that cell ran; None means scan every marker in 4d.
MAX_MARKERS = globals().get("MAX_MARKERS", None)

# --- 4a. Are the nulls on disk, or created downstream? ---
# If beta_marker is populated here, fitting is fine and the nulls arrive later
# (FDR, compile, or the shared-file schema) — a completely different fix.
print(f"=== 4a. {SPEC} / {CANCER}")
results_path = os.path.join(BIOMARKER_PATH, f"IPTW_runs_{SPEC}", f"{CANCER}_results.parquet")
if not os.path.isfile(results_path):
    print(f"  missing: {results_path}")
    t1 = None
else:
    res = pl.read_parquet(results_path)
    t1 = res.filter(pl.col("track") == 1)
    print(f"  track-1 rows: {t1.height}   "
          f"weight_types: {t1['weight_type'].unique().to_list() if t1.height else '-'}")
    for c in ("beta_marker", "p_marker", "FDR_marker", "significant_marker"):
        if c in t1.columns and t1.height:
            s = t1[c]
            nan = int(s.is_nan().sum()) if s.dtype in (pl.Float32, pl.Float64) else 0
            print(f"    {c:<20} null={s.null_count():<6} nan={nan:<6} dtype={s.dtype}")
    if t1.height:
        show = [c for c in ("marker", "beta_marker", "p_marker",
                            "n_marker_pos", "events_marker_pos") if c in t1.columns]
        print(t1.select(show).head(5))

# --- 4b. Which covariates are degenerate on the ICI frame? ---
full = pl.read_parquet(os.path.join(BIOMARKER_PATH, f"IPTW_df_{SPEC}.parquet"))
ici = full.filter(pl.col("PX_on_ICI") == 1)
print(f"\n=== 4b. ICI-only frame: {ici.height} rows (of {full.height})")

base_vars = (["GENDER", "AGE_AT_TREATMENTSTART"]
             + sorted(c for c in full.columns if c.startswith("LINE_"))
             + [c for c in full.columns if c.upper().startswith("PANEL_VERSION_")]
             + [c for c in full.columns if c.startswith("CANCER_TYPE_")])

constant_on_ici = []
for c in base_vars:
    if c not in ici.columns:
        print(f"  {c:<44} ABSENT")
        continue
    n_unique = ici[c].n_unique()
    total = float(ici[c].cast(pl.Float64, strict=False).sum() or 0)
    if n_unique <= 1:
        constant_on_ici.append(c)
    print(f"  {c:<44} n_unique={n_unique:<4} sum={total:<10.0f}"
          f"{'  <-- CONSTANT' if n_unique <= 1 else ''}")

ici_base_vars = [c for c in base_vars if c in ici.columns and ici[c].n_unique() > 1]
print(f"\n  usable on ICI frame: {len(ici_base_vars)}/{len(base_vars)}"
      f"{'   constant: ' + ', '.join(constant_on_ici) if constant_on_ici else ''}")

# --- 4c. The generalizability weights ---
# 1/ps on subset-recalibrated scores: a near-zero score is a huge weight, and
# robust=True builds its sandwich estimator from these.
eps = 1e-6
ps_ici = ici["ICI_prediction"].clip(eps, 1 - eps).to_numpy()
w_gen = 1.0 / ps_ici
low_gen, high_gen = np.percentile(w_gen, IPTW_TRUNC_PCT)
w_trunc = np.clip(w_gen, low_gen, high_gen)
ici = ici.with_columns(pl.Series("IPTW_GEN", w_trunc))
ess = w_trunc.sum() ** 2 / (w_trunc ** 2).sum()
print(f"\n=== 4c. IPTW_GEN weights")
print(f"  ICI_prediction: min={ps_ici.min():.6f} max={ps_ici.max():.6f}")
print(f"  raw 1/ps      : min={w_gen.min():.2f} max={w_gen.max():.2f}")
print(f"  truncated     : min={w_trunc.min():.2f} max={w_trunc.max():.2f}")
print(f"  non-finite    : {int((~np.isfinite(w_trunc)).sum())}")
print(f"  ESS           : {ess:.0f} of {len(w_trunc)} "
      f"({ess / max(len(w_trunc), 1) * 100:.1f}%)")

# --- 4d. One fit, uncaught ---
# _safe_fit swallows the exception and keeps only str(e); this shows the traceback.
tags = ("_SNV", "_SV", "_FUSION", "_DEL", "_AMP")
excluded = set(["DFCI_MRN", "tt_death", "death", "PX_on_ICI", "ICI_prediction"] + base_vars)
markers = [c for c in full.columns
           if c not in excluded and any(t in c.upper() for t in tags)]
# Same cap as section 3: scanning every marker for support is the slow step here
# too, and one supported marker is all 4d needs.
if MAX_MARKERS is not None:
    markers = markers[:MAX_MARKERS]
supported = [m for m in markers
             if marker_has_ici_only_support(full, m, min_pos=MIN_MARKER_POS_ICI_ONLY,
                                            min_events=MIN_EVENTS_PER_MARKER_GROUP)]
cap_note = "" if MAX_MARKERS is None else f" (capped at {MAX_MARKERS})"
print(f"\n=== 4d. markers with Track 1 support: {len(supported)}/{len(markers)}{cap_note}")

if not supported:
    print("  >>> No marker has ICI-only support, so the Track 1 screen never ran.")
else:
    for weights_col in ("IPTW_GEN", None):
        print(f"\n  --- fit: marker={supported[0]}, weights={weights_col} ---")
        try:
            out = _fit_track1_marker(ici, supported[0], ici_base_vars, weights_col)
            for k, v in out.items():
                bad = isinstance(v, float) and not np.isfinite(v)
                print(f"    {k:<22} {v}{'   <-- NON-FINITE' if bad else ''}")
        except Exception:
            traceback.print_exc()

## Reading the output

| What section 3 shows | What it means | What to do |
|---|---|---|
| `NON-NUMERIC base_vars` naming a column | That column empties every model frame | Exclude it from the covariate filter in `run_IPTW_analysis.main()`, or dummy-code it in `generate_IPTW_df` |
| `Rows surviving ... 0/N` | Confirmed: no fit could have succeeded | As above — then re-run stage 5 |
| Trial fit raises with something else | A real modelling failure, not a plumbing one | Read the traceback; `_safe_fit` would have hidden it |
| `IPTW_df: 0 patients` | The break is upstream | Re-run stage 4 (`generate_IPTW_df`) and check its join funnel |
| `Track 2 markers with support: 0/N` | Cohort too small or too imbalanced for the support thresholds | Check `MIN_MARKER_POS_PER_ARM` / `MIN_EVENTS_PER_MARKER_GROUP` against the cohort sizes printed above |
| Everything numeric, markers have support, trial fits OK | The inputs are sound now | Re-run stage 5 in 05b (`RUN_COX = True`, `SKIP_IF_DONE = False`) |

After a re-run, stage 5 can no longer end this way quietly: `_run_marker_screen` raises with a tally
of the distinct fit errors rather than writing an empty file, and
`assert_model_covariates_numeric` fails before any screen starts.

### Reading section 4 (Track 1 nulls)

| What 4a–4d shows | What it means | What to do |
|---|---|---|
| 4a `beta_marker` populated, only `significant_marker` all-False | Fits are fine — Track 1 simply has no FDR-significant hits. The ICI arm is a fraction of the cohort, so this is a plausible real result | Nothing to fix |
| 4a `beta_marker` null/NaN on disk | Confirmed: the nulls come from fitting | Continue to 4b–4d |
| 4a rows exist but `beta_marker` is absent as a column | The results-frame schema, not the fit | Check `TRACK1_RESULT_COLS` against `_fit_track1_marker`'s return keys |
| 4b any covariate flagged `CONSTANT` | Degenerate design on the ICI frame — a singular Hessian for every marker | `_run_track1_screen` recomputes `base_vars` against `ici_only_df`; confirm that filter is present and reached |
| 4c huge `raw 1/ps` max, or ESS a small fraction of N | Weight instability from near-zero recalibrated propensity scores | Tighten `IPTW_TRUNC_PCT`, or raise the `eps` floor on `ICI_prediction` |
| 4c non-finite > 0 | Weights themselves are broken | Fix upstream — `robust=True` builds its sandwich estimator from these |
| 4d fit returns finite values | The single fit works; the failure is specific to the screen's frame or weighting | Compare `ici_base_vars` here against what the screen passes |
| 4d traceback | The real error, unswallowed | Fix per the traceback |
| 4d `no marker has ICI-only support` | The screen never ran; rows would be absent, not null | Check `MIN_MARKER_POS_ICI_ONLY` / `MIN_EVENTS_PER_MARKER_GROUP` against the ICI arm size in 4b |